# VOS + Wan2.2 Animate Replacement — primeiro teste real
Execute célula por célula. A preparação pesada acontece **antes** de exportar o projeto temporário do Render. Se o diagnóstico bloquear, pare: T4 16 GB não foi comprovada. Nada aqui autoriza plano/GPU pagos ou outro modelo.


In [ ]:
import sys,pathlib,shutil,subprocess,json
from google.colab import files

## 1. Diagnóstico completo, sem uploads ou downloads
Selecione GPU gratuita no menu Ambiente de execução. Confere CUDA, contexto, VRAM total/livre, RAM disponível, disco e BF16 nativo. Aprovação só permite tentar. Se bloquear, guarde o ZIP pequeno de diagnóstico; não execute outras células.


In [ ]:
PREFLIGHT_OK=False
"""Zero-network runtime diagnosis. This source is embedded in the first notebook cell."""
import datetime
import json
from pathlib import Path
import platform
import shutil
import sys
import subprocess


def diagnose(torch_module=None,psutil_module=None,root='/content'):
    report={'schema':'vos-preflight/2','timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'python':sys.version.split()[0],'platform':platform.system(),'gpu':'CUDA indisponivel',
        'vram_gib':0,'free_vram_gib':0,'available_ram_gib':0,'free_disk_gib':0,
        'cuda_runtime':None,'compute_capability':None,'bf16_nativo':False,'cuda_context_ok':False,'failures':[]}
    try:
        if psutil_module is None:import psutil as psutil_module
        report['available_ram_gib']=round(psutil_module.virtual_memory().available/2**30,2)
        report['free_disk_gib']=round(shutil.disk_usage(root).free/2**30,2)
    except Exception as error:report['failures'].append('Nao foi possivel medir RAM/disco: '+str(error)[:300])
    try:
        if torch_module is None:import torch as torch_module
        report['torch_version']=str(getattr(torch_module,'__version__','indisponivel'))
        report['cuda_runtime']=getattr(getattr(torch_module,'version',None),'cuda',None)
        gpu_present=bool(torch_module.cuda.is_available())
        if gpu_present:
            device=torch_module.cuda.get_device_properties(0);report['gpu']=device.name
            report['vram_gib']=round(device.total_memory/2**30,2)
            capability=torch_module.cuda.get_device_capability(0);report['compute_capability']=list(capability)
            free,total=torch_module.cuda.mem_get_info(0) # initializes CUDA context, not inference
            report['free_vram_gib']=round(free/2**30,2);report['cuda_context_ok']=True
            try:supported=torch_module.cuda.is_bf16_supported(including_emulation=False)
            except TypeError:supported=torch_module.cuda.is_bf16_supported()
            report['bf16_nativo']=bool(supported and capability[0]>=8)
        else:report['failures'].append('CUDA indisponivel: selecione uma GPU gratuita ou encerre a tentativa.')
    except Exception as error:report['failures'].append('Falha ao inicializar CUDA/GPU: '+str(error)[:300])
    if not report['cuda_runtime']:report['failures'].append('PyTorch sem runtime CUDA utilizavel.')
    if report['vram_gib']<32 or report['free_vram_gib']<32:report['failures'].append('Recursos insuficientes: precisa de 32 GiB de VRAM total e livre nesta tentativa preventiva.')
    if report['available_ram_gib']<24:report['failures'].append('Recursos insuficientes: RAM disponivel abaixo de 24 GiB.')
    if report['free_disk_gib']<85:report['failures'].append('Recursos insuficientes: disco livre abaixo de 85 GiB.')
    if not report['bf16_nativo']:report['failures'].append('GPU sem BF16 nativo para esta configuracao.')
    if not report['cuda_context_ok']:report['failures'].append('Contexto CUDA nao foi confirmado.')
    if shutil.which('nvidia-smi'):
        try:report['driver_version']=subprocess.check_output(['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader'],text=True,timeout=3).strip()[:200]
        except (OSError,subprocess.SubprocessError):report['driver_version']='indisponivel'
    report['passou_bloqueio_preventivo']=not report['failures']
    report['observacao']='Passar permite tentar; nao comprova Wan no Colab gratuito e nao autoriza GPU paga.'
    return report


def check_resources():
    global PREFLIGHT_OK
    PREFLIGHT_OK=False
    report=diagnose(root='/content')
    print(json.dumps(report,ensure_ascii=False,indent=2))
    pathlib.Path('/content/VOS_GPU_DIAGNOSTICO.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
    if not report['passou_bloqueio_preventivo']:
        import zipfile
        with zipfile.ZipFile('/content/VOS_PREFLIGHT_LOGS.zip','w') as z:
            z.writestr('stage.json',json.dumps({'stage':'diagnostico','status':'blocked'}))
            z.write('/content/VOS_GPU_DIAGNOSTICO.json','VOS_GPU_DIAGNOSTICO.json')
        print('BLOQUEADO. Baixe VOS_PREFLIGHT_LOGS.zip. Nao envie arquivos nem instale modelos.')
        try:files.download('/content/VOS_PREFLIGHT_LOGS.zip')
        except Exception:print('Baixe pelo painel Arquivos.')
        raise RuntimeError('; '.join(report['failures']))
    PREFLIGHT_OK=True
    return report
check_resources()
print('Permite tentar. Wan no Colab gratuito ainda nao foi comprovado.')

## 2. Carregar ferramentas da revisão fixada
Não edite o commit. Nenhum vídeo é enviado aqui. Falhas seguintes geram logs sanitizados; backups de seleção/máscaras são separados.


In [ ]:
if not globals().get('PREFLIGHT_OK',False):raise RuntimeError('Diagnostico bloqueado. Pare aqui; nao envie arquivos nem baixe pesos.')
import urllib.request
VOS_REF='f05d966572a2fef7f6b4b46a2f0ade27e5362fd5'
TOOLS=pathlib.Path('/content/vos_tools/frame_studio')
(TOOLS/'static').mkdir(parents=True,exist_ok=True)
try:
    for name in ('__init__.py','wan_bridge.py','target_masks.py','wan_target.py','colab_support.py','wan_audit.py','preflight.py','recovery.py','smoke_test.py','static/target_editor.js'):
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/aruanmf446-hub/Vos/{VOS_REF}/frame_studio/{name}',str(TOOLS/name))
except Exception as error:
    import zipfile
    with zipfile.ZipFile('/content/VOS_BOOTSTRAP_LOGS.zip','w') as z:
        z.writestr('stage.json',json.dumps({'stage':'ferramentas','status':'failed','error_type':type(error).__name__}))
        z.write('/content/VOS_GPU_DIAGNOSTICO.json','VOS_GPU_DIAGNOSTICO.json')
    print('Falha nas ferramentas. Baixe VOS_BOOTSTRAP_LOGS.zip e reexecute esta celula; nao continue.')
    raise
sys.path.insert(0,str(TOOLS.parent))
from frame_studio.colab_support import Session,Pipeline,install_error_hook
from frame_studio.recovery import export_progress,restore_progress
from frame_studio.target_masks import show_editor,require_review
WORK=pathlib.Path('/content/vos_wan_job')
SESSION=Session('/content/vos_logs',work=WORK,download=files.download)
FLOW=Pipeline(SESSION)
install_error_hook(SESSION,get_ipython())
shutil.copyfile('/content/VOS_GPU_DIAGNOSTICO.json',SESSION.folder/'VOS_GPU_DIAGNOSTICO.json')
BRIDGE=TOOLS/'wan_bridge.py'
def backup(stage):
    output=export_progress(WORK,WORK.parent/f'VOS_PROGRESSO_{stage}.zip')
    print('Guarde este backup com o ZIP original e a referencia. Nao importar o backup no VOS:',output.name)
    files.download(str(output))


## 3. Instalar, auditar e preparar o teste mínimo sintético
Só com diagnóstico aprovado: instala Wan oficial, confere imports e funções de geometria/rosto antes de baixar pesos ModelScope, verifica hashes e executa transporte sintético 24/30/60 FPS. O teste sintético não carrega SAM2/Wan e não gera uma entrega real. Pode demorar; não crie o projeto Render ainda.


In [ ]:
if not globals().get('PREFLIGHT_OK',False):raise RuntimeError('Diagnostico bloqueado. Pare aqui; nao envie arquivos nem baixe pesos.')
FLOW.begin('install',PREFLIGHT_OK)
import os
os.environ.update(HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',HF_DATASETS_OFFLINE='1')
WAN_REV='1ea34ff48f87168174e12956e200b1d908b1c5ff'
WAN=pathlib.Path('/content/Wan2.2')
if not WAN.exists():SESSION.run(['git','clone','--no-checkout','https://github.com/Wan-Video/Wan2.2.git',str(WAN)])
SESSION.run(['git','-C',str(WAN),'checkout','--detach',WAN_REV])
SESSION.run([sys.executable,'-m','pip','install','--no-build-isolation','-r',str(WAN/'requirements.txt'),'-r',str(WAN/'requirements_animate.txt'),'moviepy==1.0.3','diffusers==0.35.1'])
SESSION.run([sys.executable,'-m','pip','install','modelscope'])
CKPT=pathlib.Path('/content/Wan2.2-Animate-14B')
WEIGHTS_MANIFEST=pathlib.Path('/content/wan22_modelscope_manifest.json')
urllib.request.urlretrieve(f'https://raw.githubusercontent.com/aruanmf446-hub/Vos/{VOS_REF}/colab/wan22_modelscope_manifest.json',str(WEIGHTS_MANIFEST))
import json
weights=json.loads(WEIGHTS_MANIFEST.read_text())
if shutil.disk_usage('/content').free < sum(entry['size'] for entry in weights['files'])+15*2**30:
    raise RuntimeError('Disco insuficiente para pesos selecionados e arquivos de trabalho')
SESSION.run([sys.executable,str(TOOLS/'wan_audit.py'),'--wan',str(WAN),'--dependencies'],label='Conferir funcoes oficiais e dependencias antes dos pesos')
SESSION.run([sys.executable,str(TOOLS/'smoke_test.py'),'--report',str(SESSION.folder/'synthetic_smoke.json')],label='Teste sintetico de transporte, SEM inferencia')
from modelscope import snapshot_download
snapshot_download(weights['model_id'],revision=weights['revision'],local_dir=str(CKPT),allow_patterns=[entry['path'] for entry in weights['files']])
SESSION.run([sys.executable,str(BRIDGE),'verify-weights','--checkpoint',str(CKPT),'--manifest',str(WEIGHTS_MANIFEST)])
print('Wan revision:',WAN_REV)
SESSION.snapshot()
FLOW.complete('install')
print('Runtime preparado. Agora exporte uma cena NOVA e curta no VOS para reduzir risco de expiracao no Render.')

## 4. Agora crie o projeto no VOS e envie uma cena curta
Com runtime pronto, exporte 1–3 segundos, UMA cena, cadência Original. Guarde o ZIP original e a referência no computador. Mantenha o mesmo navegador/projeto VOS: o padrão é 2 horas sem acesso, e reinício/hibernação do Render pode apagar tudo antes. O backup Colab não recupera o projeto Render expirado nem permite trocar manifestos.


In [ ]:
if not globals().get('PREFLIGHT_OK',False):raise RuntimeError('Diagnostico bloqueado. Pare aqui; nao envie arquivos nem baixe pesos.')
FLOW.begin('prepare',PREFLIGHT_OK,requires=('install',))
print('Escolha ZIP original do VOS')
u=files.upload();ZIP=pathlib.Path('/content')/next(n for n in u if n.lower().endswith('.zip'));del u
print('Escolha imagem PNG/JPG do novo personagem')
u=files.upload();REF=pathlib.Path('/content')/next(n for n in u if n.lower().endswith(('.png','.jpg','.jpeg')));del u
import tempfile
WORK=pathlib.Path(tempfile.mkdtemp(prefix='vos_wan_job_',dir='/content'))
SESSION.work=WORK
SESSION.run([sys.executable,str(BRIDGE),'prepare','--zip',str(ZIP),'--reference',str(REF),'--work',str(WORK)],label='Preparar cena real sem IA')
FLOW.complete('prepare')
backup('preparacao')
from IPython.display import display,Video
display(Video(str(WORK/'driving.mp4'),embed=True))

## 5. Seleção opcional ou recuperação de seleção/máscaras
Desligado por padrão. Para recuperar, envie também um backup VOS_PROGRESSO com o **mesmo ZIP original** preparado acima. Isso recupera pontos/máscaras; não retoma inferência interrompida. Um backup de outro projeto/exportação será recusado.


In [ ]:
if not globals().get('PREFLIGHT_OK',False):raise RuntimeError('Diagnostico bloqueado. Pare aqui; nao envie arquivos nem baixe pesos.')
FLOW.begin('selection',PREFLIGHT_OK,requires=('prepare',))
USE_TARGET_SELECTION=False #@param {type:"boolean"}
RESTORE_PROGRESS=False #@param {type:"boolean"}
if RESTORE_PROGRESS:
    print('Escolha VOS_PROGRESSO_*.zip (nao o ZIP de edicoes)')
    u=files.upload();archive=pathlib.Path('/content')/next(n for n in u if n.lower().endswith('.zip'));del u
    restored=restore_progress(WORK,archive)
    USE_TARGET_SELECTION=(restored!='no_selection')
    print('Recuperacao:',restored,'; inferencia nao foi retomada.')
elif USE_TARGET_SELECTION:
    show_editor(WORK,'prompts')
elif (WORK/'target.json').exists():
    raise RuntimeError('Selecao ja ativa. Use o modo selecionado ou um trabalho novo; nao apague recibos manualmente.')
FLOW.complete('selection')
backup('selecao')

## 6. SAM2 e revisão, somente se a opção foi ativada
Pontos verdes no alvo, vermelhos na outra pessoa; corrija máscaras com pincel/borracha e revise cada quadro. Alertas bloqueiam a aprovação. Máscaras recuperadas e já revisadas, vinculadas ao mesmo manifesto, dispensam novo rastreamento. Não há garantia automática de identidade.


In [ ]:
if not globals().get('PREFLIGHT_OK',False):raise RuntimeError('Diagnostico bloqueado. Pare aqui; nao envie arquivos nem baixe pesos.')
FLOW.begin('track',PREFLIGHT_OK,requires=('install','prepare','selection'))
if USE_TARGET_SELECTION:
    try:require_review(WORK);already_reviewed=True
    except (ValueError,OSError):already_reviewed=False
    if not already_reviewed:
        SESSION.run([sys.executable,str(TOOLS/'wan_target.py'),'track','--work',str(WORK),'--wan',str(WAN),'--checkpoint',str(CKPT)],label='SAM2 alvo explicito experimental')
        show_editor(WORK,'review')
    else:print('Mascaras revisadas do mesmo ZIP recuperadas; nenhuma inferencia retomada.')
    backup('mascaras_revisadas')
FLOW.complete('track')

## 7. Pré-processar pose/rosto/máscara
Ainda não é inferência generativa. Se falhar, baixe VOS_LOGS.zip; corrija e reexecute a etapa. Células seguintes não podem contornar uma falha.


In [ ]:
if not globals().get('PREFLIGHT_OK',False):raise RuntimeError('Diagnostico bloqueado. Pare aqui; nao envie arquivos nem baixe pesos.')
FLOW.begin('preprocess',PREFLIGHT_OK,requires=('install','prepare','track'))
SESSION.run([sys.executable,str(BRIDGE),'preprocess','--work',str(WORK),'--wan',str(WAN),'--checkpoint',str(CKPT)],label='Preprocessar pose rosto mascara')
FLOW.complete('preprocess')

## 8. Revisar materiais antes de iniciar o modelo
Assista aos vídeos inteiros. Rosto e esqueleto devem seguir o alvo correto. Branco na máscara é região alterada. SIM registra a revisão, não corrige seleção incorreta.


In [ ]:
if not globals().get('PREFLIGHT_OK',False):raise RuntimeError('Diagnostico bloqueado. Pare aqui; nao envie arquivos nem baixe pesos.')
FLOW.begin('review',PREFLIGHT_OK,requires=('preprocess',))
(WORK/'mask_review.json').unlink(missing_ok=True)
for name in ('driving.mp4','wan_processed/src_mask.mp4','wan_processed/src_pose.mp4','wan_processed/src_face.mp4'):
    print('Revisar:',name);display(Video(str(WORK/name),embed=True))
if input('Somente o alvo correto, mascara/pose/rosto corretos em toda cena? Digite SIM: ').strip().upper()!='SIM':
    raise RuntimeError('Geracao cancelada; corrija selecao/mascaras e reexecute as etapas.')
SESSION.run([sys.executable,str(BRIDGE),'review-mask','--work',str(WORK),'--confirm','SIM'],label='Confirmar revisao humana')
FLOW.complete('review')
backup('antes_da_inferencia')

## 9. Inferência Wan real — só aqui o modelo gera
A primeira tentativa deve manter o perfil experimental desligado. OOM/queda não têm promessa de retomada. Garanta os downloads dos backups antes de executar. Não pule o diagnóstico e não contrate GPU paga.


In [ ]:
if not globals().get('PREFLIGHT_OK',False):raise RuntimeError('Diagnostico bloqueado. Pare aqui; nao envie arquivos nem baixe pesos.')
FLOW.begin('infer',PREFLIGHT_OK,requires=('install','prepare','review'))
LOW_MEMORY_EXPERIMENTAL=False #@param {type:"boolean"}
command=[sys.executable,str(BRIDGE),'infer','--work',str(WORK),'--wan',str(WAN),'--checkpoint',str(CKPT)]
if LOW_MEMORY_EXPERIMENTAL:command.append('--low-memory')
SESSION.run(command,label='Wan Animate Replacement real')
FLOW.complete('infer')
display(Video(str(WORK/'wan_generated.mp4'),embed=True))

## 10. Recompor e voltar ao mesmo projeto VOS
Só após inferência bem-sucedida. Importe apenas VOS_EDITADO.zip no projeto original, valide e revise antes de aplicar e montar MP4. Se expirou, não altere manifesto para forçar importação: será necessário novo projeto/exportação e ciclo compatível.


In [ ]:
if not globals().get('PREFLIGHT_OK',False):raise RuntimeError('Diagnostico bloqueado. Pare aqui; nao envie arquivos nem baixe pesos.')
FLOW.begin('finish',PREFLIGHT_OK,requires=('infer',))
OUT=pathlib.Path('/content/VOS_EDITADO.zip')
SESSION.run([sys.executable,str(BRIDGE),'finish','--work',str(WORK),'--output',str(OUT)],label='Recompor ZIP real')
FLOW.complete('finish')
files.download(str(OUT))
files.download(str(SESSION.bundle()))